<div align="center" style="text-align: center; padding: 60px 0;">

<h1 style="text-align: center;">Monitoring glacier retreat and glacial lake growth in the Mount Everest massif with Sentinel-2 (2017–2025)</h1>

<br>

<h2 style="text-align: center;">Remote Sensing &amp; Computer Vision</h2>

<br><br>

<p style="text-align: center; font-size: 18px;"><b>Authors:</b> Luka Rondal &amp; Tanguy Mandrillon</p>

<p style="text-align: center; font-size: 18px;">2026</p>

</div>


## **Context**
Glaciers of the Everest massif (Khumbu, Rongbuk, Kangshung, Ngozumpa, 
Imja–Lhotse Shar...) are losing mass because of rising temperatures. Two visible 
consequences are the shrinking of clean ice/snow-covered surfaces and the growth of 
pro-glacial lakes such as Imja Tsho, which increases the risk of glacial lake outburst 
floods (GLOF) for the valleys downstream. The goal is to quantify these two phenomena 
over the last decade. 

## 1. Bibliography
In this section, we will define the scope and make sope assumption for our analysis.

### **Geography analysis**
We will chose a glacier and a lake to analyze.

| Type | Name | Description | Source |
|:---|:---|:---|:---|
| Glacier | Khumbu Glacier | Located in north-eastern Nepal, between Mount Everest and the Lhotse–Nuptse ridge. It is one of the highest glaciers in the world and includes the Khumbu Icefall on the standard Everest climbing route. Its lower tongue is largely debris-covered. | [Wikipedia – Khumbu Glacier](https://en.wikipedia.org/wiki/Khumbu_Glacier) |
| Lake | Imja Tsho | Pro-glacial lake at about 5,000 m altitude, at the terminus of the Imja and Lhotse Shar glaciers, south of Everest. It formed from melt ponds in the second half of the 20th century and is one of the fastest-growing glacial lakes in the Himalaya, with a significant GLOF risk for the valleys downstream. | [Wikipedia – Imja Tsho](https://en.wikipedia.org/wiki/Imja_Tsho) |

### **Spectral indices: NDSI and NDWI**

To separate our classes we use two **normalized difference indices**. A normalized difference index compares the reflectance of a pixel in two spectral bands where the target surface behaves very differently:

$$\text{Index} = \frac{\rho_1 - \rho_2}{\rho_1 + \rho_2} \in [-1, 1]$$

The normalization makes the index less sensitive to illumination differences (slopes, shadows, sun angle), which matters in a steep mountain area like the Everest massif.

**[NDSI – Normalized Difference Snow Index](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/ndsi/)**

$$NDSI = \frac{B03 - B11}{B03 + B11}$$

- `B03` = green (560 nm, 10 m) and `B11` = short-wave infrared, SWIR (1610 nm, 20 m).
- **What it shows:** snow and ice reflect a lot of visible light but absorb strongly in the SWIR, so they have a **high NDSI**. Rock, soil and debris reflect about the same amount in both bands, so their NDSI is **close to 0 or negative**. Clouds are bright in both bands, which gives a low NDSI and lets the index separate snow from most clouds.
- **Threshold:** a pixel is classified as snow/ice when **NDSI > 0.42** (Hall et al., 1995).
- **Limitation:** NDSI cannot tell snow from clean glacier ice (both are high), so we merge them into one **snow/ice** class. Ice covered by debris looks like rock and is **not** detected.

**[NDWI – Normalized Difference Water Index](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/ndwi/)**

$$NDWI = \frac{B03 - B08}{B03 + B08}$$

- `B03` = green (560 nm, 10 m) and `B08` = near infrared, NIR (842 nm, 10 m).
- **What it shows:** open water absorbs almost all NIR light, so it has a **positive NDWI**. Vegetation and bare soil reflect more NIR than green, so they have a **negative NDWI**.
- **Threshold:** the threshold depends on the source. McFeeters (1996) uses **NDWI > 0** for water, while Sentinel Hub states that *"index values greater than 0.5 usually correspond to water bodies"*. In a glacier area, a threshold that is too low is risky: snow and clean ice can also have slightly positive NDWI values. A threshold that is too high misses turbid glacial lakes, which carry a lot of sediment. We will therefore **tune the threshold between 0.2 and 0.5** by checking the result on true-colour images of Imja Tsho.
- **Why we need it:** water also has a **high NDSI**, since it absorbs SWIR as well. NDSI alone would classify lakes such as Imja Tsho as ice, so NDWI is used to **separate lakes from ice**.

### **Classification rules**

Each pixel gets one of 4 classes. The order of the tests matters:

| Order | Condition | Class |
|:---:|:---|:---|
| 1 | SCL = 0, 3, 8, 9 or 10 (no data, cloud shadow, clouds, cirrus) | Cloud / no data |
| 2 | NDWI > threshold (0.2 – 0.5, tuned) | Water (glacial lake) |
| 3 | NDSI > 0.4 | Snow / ice |
| 4 | otherwise | Rock / debris |

Water is tested **before** snow/ice because lakes have both a high NDWI and a high NDSI. Area per class = number of pixels × 400 m² (20 m pixels).

## 2. Parameters

All the **variable inputs** of the pipeline are gathered in the next cell. To run the analysis on another area, period or threshold, only this cell has to be modified, then the whole notebook can be re-run.

| Parameter | Value | Justification |
|:---|:---|:---|
| `BBOX_MASSIF` | 86.70–87.15° E, 27.80–28.20° N | Whole Everest massif (Khumbu, Rongbuk, Kangshung, Ngozumpa, Imja–Lhotse Shar), obtained with [bboxfinder.com](http://bboxfinder.com) |
| `BBOX_KHUMBU` | 86.79–86.94° E, 27.92–28.02° N | Khumbu Glacier, from its terminus (27.93° N, 86.81° E) up to the Khumbu Icefall and the Western Cwm |
| `BBOX_IMJA_TSHO` | 86.90–86.97° E, 27.88–27.92° N | Imja Tsho (27.90° N, 86.93° E, about 2 km × 650 m) and the tongues of the Imja and Lhotse Shar glaciers that feed it |
| `YEARS` | 2017 → 2025 | One image per year, to follow the trend over the last decade |
| `SEASON_START` / `SEASON_END` | 1 Oct → 30 Nov | Post-monsoon: fewest clouds and minimal seasonal snow, so the snow/ice surface is close to the glacier surface |
| `MAX_CLOUD_COVER` | 20 % | Maximum cloud cover of the selected acquisition |
| `RESOLUTION` | 20 m | Native resolution of B11 (SWIR); keeps the whole massif under the Sentinel Hub limit of 2500 × 2500 px |
| `NDSI_THRESHOLD` | 0.42 | Snow/ice threshold (see section 1) |
| `NDWI_THRESHOLD` | 0.3 | Water threshold, starting value to tune between 0.2 and 0.5 (see section 1) |
| `SCL_MASK_CLASSES` | 0, 3, 8, 9, 10 | No data, cloud shadow, clouds (medium/high probability), thin cirrus |
| `MAX_MASKED_FRACTION` | 30 % | Years with more masked pixels are flagged as unreliable |

The Copernicus credentials are **not** written in the notebook: they are imported from the separate file `myCopernicusCredentials.py`.

In [1]:
# --- Areas of interest (lon_min, lat_min, lon_max, lat_max) in WGS84 (EPSG:4326)
BBOX_MASSIF = [86.70, 27.80, 87.15, 28.20]     # Whole Everest massif, from bboxfinder.com
BBOX_KHUMBU = [86.79, 27.92, 86.94, 28.02]     # Khumbu Glacier: from the terminus (27.93 N, 86.81 E) to the Western Cwm
BBOX_IMJA_TSHO = [86.90, 27.88, 86.97, 27.92]  # Imja Tsho (27.90 N, 86.93 E) + Imja / Lhotse Shar glacier tongues

AOIS = {
    "Everest massif": BBOX_MASSIF,
    "Khumbu Glacier": BBOX_KHUMBU,
    "Imja Tsho": BBOX_IMJA_TSHO,
}

# --- Period
YEARS = list(range(2017, 2026))  # 2017 -> 2025 (range end is excluded)
SEASON_START = "10-01"           # MM-DD, post-monsoon window
SEASON_END = "11-30"             # MM-DD

# --- Image selection
MAX_CLOUD_COVER = 0.20  # Max cloud cover of the acquisition (0 to 1)
RESOLUTION = 20         # Pixel size in meters (B11 native resolution)

# --- Classification thresholds (see section 1)
NDSI_THRESHOLD = 0.42  # snow/ice if NDSI > NDSI_THRESHOLD
NDWI_THRESHOLD = 0.3   # water if NDWI > NDWI_THRESHOLD (to tune between 0.2 and 0.5)

# --- Cloud / no-data mask from the Scene Classification Layer (SCL)
SCL_MASK_CLASSES = [0, 3, 8, 9, 10]  # no data, cloud shadow, cloud medium/high proba, thin cirrus
MAX_MASKED_FRACTION = 0.30           # years with more masked pixels are flagged

# --- Bands requested to Sentinel Hub (Sentinel-2 L2A)
BANDS = ["B02", "B03", "B04", "B08", "B11", "SCL"]

# --- Derived values
PIXEL_AREA_KM2 = (RESOLUTION * RESOLUTION) / 1e6  # 20 m x 20 m = 400 m2 = 0.0004 km2
TIME_INTERVALS = {year: (f"{year}-{SEASON_START}", f"{year}-{SEASON_END}") for year in YEARS}

for name, bbox in AOIS.items():
    print(f"{name}: {bbox}")
print(f"Years: {YEARS[0]} -> {YEARS[-1]} ({len(YEARS)} images)")
print(f"Season window: {SEASON_START} -> {SEASON_END}")
print(f"Pixel area: {PIXEL_AREA_KM2} km2")

Everest massif: [86.7, 27.8, 87.15, 28.2]
Khumbu Glacier: [86.79, 27.92, 86.94, 28.02]
Imja Tsho: [86.9, 27.88, 86.97, 27.92]
Years: 2017 -> 2025 (9 images)
Season window: 10-01 -> 11-30
Pixel area: 0.0004 km2


Next we configure the connection to the Copernicus Data Space Ecosystem (Sentinel Hub API) and check that the requested image stays under the Sentinel Hub size limit (2500 × 2500 px).

In [2]:
import sys
import importlib
import sentinelhub
import myCopernicusCredentials
importlib.reload(myCopernicusCredentials)

# --- Sentinel Hub configuration (Copernicus Data Space Ecosystem)
config = sentinelhub.SHConfig()
config.sh_client_id = myCopernicusCredentials.client_id
config.sh_client_secret = myCopernicusCredentials.client_secret
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
config.sh_base_url = "https://sh.dataspace.copernicus.eu"

# --- Bounding boxes and image sizes
aoi_bboxes = {}
aoi_sizes = {}
for name, bbox in AOIS.items():
    aoi_bboxes[name] = sentinelhub.BBox(bbox=bbox, crs=sentinelhub.CRS.WGS84)
    aoi_sizes[name] = sentinelhub.bbox_to_dimensions(aoi_bboxes[name], resolution=RESOLUTION)
    print(f"{name}: {aoi_sizes[name][0]} x {aoi_sizes[name][1]} px at {RESOLUTION} m")

    if max(aoi_sizes[name]) > 2500:
        print(f"WARNING: {name} larger than 2500 px, increase RESOLUTION or reduce the bbox")

c:\Users\lulul\Documents\EPF\S2-Data\Remote Sensing & Computer Vision\TP\Lab 2 - part 2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Everest massif: 2214 x 2214 px at 20 m
Khumbu Glacier: 738 x 553 px at 20 m
Imja Tsho: 345 x 221 px at 20 m


In [11]:
import folium
import shapely

# Visual check of the areas of interest
colors = {"Everest massif": "blue", "Khumbu Glacier": "orange", "Imja Tsho": "red"}

# Base maps: OpenStreetMap by default, satellite imagery selectable in the legend
map = folium.Map(tiles="OpenStreetMap")
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                 attr="Esri World Imagery", name="Satellite (Esri)", show=False).add_to(map)

for name, bbox in AOIS.items():
    folium.GeoJson(data=shapely.geometry.box(*bbox),
                   name=f"<span style='color: {colors[name]};'>▬</span> {name}",
                   style_function=lambda x, c=colors[name]: {"color": c, "fillOpacity": 0.05}).add_to(map)

map.add_child(folium.LayerControl(collapsed=False))
map.fit_bounds([(BBOX_MASSIF[1], BBOX_MASSIF[0]), (BBOX_MASSIF[3], BBOX_MASSIF[2])])  # folium expects (lat, lon)
display(map)

## 2. Parameters
